# GenCare AI
---
Auteur:   Eva Rombouts  
Datum:    26-12-2023.  
Doel:     Ervaring opdoen met Python, NLP  
Script:   Onderzoekt een aantal opensource modellen 

Data gegenereerd met de OpenAI API.  
---

In [ ]:
#from datasets import load_dataset, Dataset, DatasetDict
#from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
#import torch
#import time
#import evaluate
import pandas as pd
import numpy as np
#from peft import LoraConfig, get_peft_model, TaskType
#from sklearn.model_selection import train_test_split

In [ ]:
# # Wanneer in colab
# from google.colab import drive
# drive.mount('/content/drive')
# gci = pd.read_csv('/content/drive/MyDrive/eColab/GenCareAI/data/gci_rapportages.csv')
# gci.info()

In [ ]:
gci = pd.read_csv('../zorgdata/gci_rapportages.csv')
gci.info()

In [ ]:
gci['onrustscore'].describe()

In [ ]:
gci['rapportage'].head()

In [ ]:
# Om een gevoel te krijgen van de lengte van de rapportages:
gci['rapportage_lengte'] = gci['rapportage'].apply(len)
gci['n_woorden'] = gci['rapportage'].str.split().apply(len)
gci[['rapportage_lengte','n_woorden']].describe()

## Modellen zoeken

In [ ]:
from transformers import pipeline

In [ ]:
model = 'wietsedv/bert-base-dutch-cased-finetuned-sentiment'
sentiment_analysis = pipeline('sentiment-analysis', model = model)

In [ ]:
# Laten we starten met de eerste week van dhr Pieterse
test_zinnen = gci.loc[(gci['cliënt_id'] == 'kamer02') & (gci['week'] == 1), 'rapportage'].tolist()
for rp in test_zinnen:
    print(rp)

In [ ]:
se_output = sentiment_analysis(test_zinnen)

for a in enumerate(se_output) :
    print(a)

Ik ben niet erg onder de indruk... Hij neigt erg naar positiviteit. 
Laten we er nog een proberen

In [ ]:
model="henryk/bert-base-multilingual-cased-finetuned-dutch-squad2"

qa_pipeline = pipeline(
    "question-answering",
    model = model,
    tokenizer = model
)

In [ ]:
for rp in test_zinnen:
    qa_resultaat = qa_pipeline({
        'context': rp,
        'question': "Hoe heet deze client?"})
#        'question': "Wat heeft de client nodig?"})
#        'question': "Hoe voelt deze client zich?"})
    print(rp)
    print(qa_resultaat['answer'])
    print('-'*100)


In [ ]:
sum_model = 'yhavinga/t5-v1.1-base-dutch-cnn-test'

In [ ]:
# Maak de summarization pipeline
summarization_pipeline = pipeline(
    "summarization",
    model=sum_model,
    tokenizer=sum_model
)

In [ ]:
# Voer de samenvatting uit
samenvatting_resultaat = summarization_pipeline(test_zinnen[1])

In [ ]:
samenvatting = samenvatting_resultaat[0]['summary_text']
print(samenvatting)

In [ ]:
for rp in test_zinnen:
    sum_resultaat = summarization_pipeline(rp, min_length = 50, max_length=70)
    smry = sum_resultaat[0]['summary_text']
    print(rp)
    print(smry)
    print('-'*100)


Wow, dat werkt best goed. 